![New York City schoolbus](schoolbus.jpg)

Photo by [Jannis Lucas](https://unsplash.com/@jannis_lucas) on [Unsplash](https://unsplash.com).
<br>

Every year, American high school students take SATs, which are standardized tests intended to measure literacy, numeracy, and writing skills. There are three sections - reading, math, and writing, each with a **maximum score of 800 points**. These tests are extremely important for students and colleges, as they play a pivotal role in the admissions process.

Analyzing the performance of schools is important for a variety of stakeholders, including policy and education professionals, researchers, government, and even parents considering which school their children should attend. 

You have been provided with a dataset called `schools.csv`, which is previewed below.

You have been tasked with answering three key questions about New York City (NYC) public school SAT performance.

In [70]:
# Re-run this cell 
import pandas as pd

# Read in the data
schools = pd.read_csv("schools.csv")

# Preview the data
schools.head()

# Start coding here...

# Schools with the best math results:
best_math_schools = schools.loc[
    (schools['average_math'] >= 800 * 0.8), ['school_name', 'average_math']
].sort_values(by='average_math', ascending=False)

# Top 10 performing schools based on combined SAT scores
schools['total_SAT'] = (
    schools['average_math'] +
    schools['average_reading'] +
    schools['average_writing']
)

top_10_schools = (
    schools
    .nlargest(10, 'total_SAT')
    .loc[:, ['school_name', 'total_SAT']]
)

# Which single borough has the largest SD in the combined SAT score? 
borough_df = (
    schools
    .groupby('borough')
    .agg(
        std_SAT = ('total_SAT', 'std'),
        num_schools = ('school_name', 'count'),
        average_SAT = ('total_SAT', 'mean'),
    )
    .round(2)
    .reset_index()
)

largest_std_dev = borough_df.nlargest(1, 'std_SAT')

# EXTRAS
# Which 10 schools have the largest absolute difference between their highest and lowest subject averages (math, reading, writing)?
schools['highest_avg'] = schools[['average_math', 'average_reading', 'average_writing']].max(axis=1)
schools['lowest_avg'] = schools[['average_math', 'average_reading', 'average_writing']].min(axis=1)
schools['avg_abs_diff'] = (
    schools['highest_avg'] - schools['lowest_avg']
)

top_10_avgdiff_schools = schools.nlargest(10, 'avg_abs_diff')

# How can NYC boroughs be grouped based on the variability of total SAT scores?
import numpy as np
condition = [
    borough_df['std_SAT'] >= 210,
    borough_df['std_SAT'] >= 170
]

outcomes = [
    'High Variability',
    'Medium Variability'
]

borough_df['Variability'] = np.select(condition, outcomes, default='Low Variability')

# Using a benchmark of 1500 total SAT score (out of 2400), what percentage of schools in each borough score at or above 1500?
benchmark_schools = (
    schools
    .loc[schools['total_SAT'] >= 1500]
    .groupby('borough')
    .agg(
        benchmark_school_count = ('total_SAT', 'count')
    )
    .reset_index()
)

benchmark_schools['total_num_schools'] = borough_df['num_schools']
benchmark_schools['percent_benchmark'] = (
    benchmark_schools['benchmark_school_count'] /
    benchmark_schools['total_num_schools'] *
    100
)

print(benchmark_schools)
print(borough_df)

         borough  benchmark_school_count  total_num_schools  percent_benchmark
0          Bronx                       2                 98           2.040816
1       Brooklyn                       6                109           5.504587
2      Manhattan                      20                 89          22.471910
3         Queens                       9                 69          13.043478
4  Staten Island                       1                 10          10.000000
         borough  std_SAT  num_schools  average_SAT         Variability
0          Bronx   150.39           98      1202.72     Low Variability
1       Brooklyn   154.87          109      1230.26     Low Variability
2      Manhattan   230.29           89      1340.13    High Variability
3         Queens   195.25           69      1345.48  Medium Variability
4  Staten Island   222.30           10      1439.00    High Variability
